# Consistency Cleaning Notebook

This notebook focuses on fixing **inconsistent data formats and values**.

### Objectives:
- Standardize categorical values
- Fix inconsistent formats:
  - Email
  - Dog Size
  - Dog Gender
  - Dog Age
- Ensure all values follow predefined valid formats
- Export cleaned dataset


## Step 1: Install Dependencies (if needed)

In [1]:
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


## Step 2: Import Libraries and Cleaning Functions

In [2]:
import sys, os
sys.path.append(os.path.abspath('../../'))

In [3]:
import pandas as pd
import numpy as np

from src.scripts.consistency import (
    fix_dog_size,
    fix_dog_gender,
    fix_email
)

## Step 3: Load Dataset

In [4]:
df = pd.read_csv('../../data/raw/dog_survey.csv')
df.columns = df.columns.str.strip().str.lower()
df.head()

,id,title,first_name,last_name,email,amount_spent_on_dog_food,dog_size,dog_gender,dog_age,unnamed: 9,unnamed: 10
0,1,Mrs,Frasquito,Dene,fdene0@dell.com,£80.05,S,M,53,NaN,NaN
1,2,Ms,Lawrence,Kardos,lkardos1@diigo.com,£66.30,XL,M,2,NaN,NaN
2,3,Rev,Lanna,Wintersgill,lwintersgill2@domainmarket.com,£25.84,L,F,71,NaN,NaN
3,4,Rev,Richmound,Kimmitt,rkimmitt3@jiathis.com,£50.76,M,M,9,NaN,NaN
4,5,Rev,Valeda,Dallender,vdallender4@theglobeandmail.com,£83.41,XL,F,23,NaN,NaN


## Step 4: Inspect Inconsistencies (Before Cleaning)

We check unique values to identify inconsistent formats.

In [5]:
print('Dog Size Values:')
print(df['dog_size'].unique())

print('\nDog Gender Values:')
print(df['dog_gender'].unique())

print('\nEmail Samples:')
print(df['email'].head(10))

print('\nDog Age Samples:')
print(df['dog_age'].head(10))

Dog Size Values:
<StringArray>
[           'S',           'XL',            'L',            'M',
           'XS',            nan,           'NO',            '-',
        'S,L,L',     'Smallish', 'Medium sized',        'large']
Length: 12, dtype: str

Dog Gender Values:
<StringArray>
[                  'M',                   'F',              'femlae',
              'Female',                'Male',                   nan,
                'male',                'MALE',          'Don’t know',
 '1 male and 1 female',              'Unkown',               'M,M,F',
                   '—',                   '-',             'Unknown',
              'female']
Length: 16, dtype: str

Email Samples:
0                    fdene0@dell.com
1                 lkardos1@diigo.com
2     lwintersgill2@domainmarket.com
3              rkimmitt3@jiathis.com
4    vdallender4@theglobeandmail.com
5                dcavell5@seesaa.net
6                ngabbetis6@admin.ch
7        fswindin7@printfriendly.com
8       

## Step 5: Fix Email Format

- Remove invalid formats
- Standardize emails

In [40]:
before = df.copy()

df = fix_email(df)

print('After email cleaning:')
df['email'].head(10)

After email cleaning:


0                    fdene0@dell.com
1                 lkardos1@diigo.com
2     lwintersgill2@domainmarket.com
3              rkimmitt3@jiathis.com
4    vdallender4@theglobeandmail.com
5                dcavell5@seesaa.net
6                ngabbetis6@admin.ch
7        fswindin7@printfriendly.com
8                    lheinsh8@i2i.jp
9                 jdaunay9@mysql.com
Name: email, dtype: str

## Step 6: Fix Dog Size

Handles values like:
- S, M, L
- smallish
- S,L,L
- NA, -, etc

In [41]:
before = df.copy()

df = fix_dog_size(df)

print(df['dog_size'].value_counts())

dog_size
Extra Large    128
Small           59
Large           43
Medium          39
Extra Small     33
Unknown          6
Name: count, dtype: int64


## Step 7: Fix Dog Gender

Handles inconsistent inputs such as:
- M, F
- MALE, female
- femlae (typo)
- M,M,F
- Unknown variants

In [42]:
before = df.copy()

df = fix_dog_gender(df)

print(df['dog_gender'].value_counts())

dog_gender
Female     150
Male       145
Unknown     13
Name: count, dtype: int64


## Step 8: Fix Dog Age (Custom Logic)

We standardize values like:
- "3,3,5" → 3
- "5 and 4" → 5
- "Less than 20" → 20
- "12+" → 12

In [43]:
s = df['dog_age'].astype(str)

# extract first number
s = s.str.extract(r'(\d+)', expand=False)

df['dog_age'] = pd.to_numeric(s, errors='coerce')

print(df['dog_age'].describe())

count    307.000000
mean      48.553746
std       28.343819
min        1.000000
25%       24.000000
50%       48.000000
75%       71.000000
max      100.000000
Name: dog_age, dtype: float64


## Step 9: Fix Amount Consistency

### Objective
Standardize and sanitize `amount_spent_on_dog_food` by removing formatting noise and invalid values.

### Rules Applied
- Remove currency symbols (e.g. £)
- Remove thousand separators (,)
- Trim whitespace
- Convert to numeric safely
- Set invalid values (≤ 0 or NaN) to missing


In [44]:
# 1. Extract raw column
s = df["amount_spent_on_dog_food"].astype(str)

# 2. Remove formatting noise (currency + commas + spaces)
s = (
    s.str.replace("£", "", regex=False)
     .str.replace(",", "", regex=False)
     .str.strip()
)

# 3. Convert to numeric (invalid parsing → NaN)
df["amount_spent_on_dog_food"] = pd.to_numeric(s, errors="coerce")

# 4. Invalidate logically incorrect values
invalid_mask = (
    df["amount_spent_on_dog_food"].isna() |
    (df["amount_spent_on_dog_food"] <= 0)
)

df.loc[invalid_mask, "amount_spent_on_dog_food"] = np.nan

# 5. Summary check (post-cleaning validation)
print("=== AMOUNT CLEANING SUMMARY ===")
print(df["amount_spent_on_dog_food"].describe())
print("Missing values:", df["amount_spent_on_dog_food"].isna().sum())


=== AMOUNT CLEANING SUMMARY ===
count    286.000000
mean      51.891049
std       29.283306
min        0.120000
25%       25.525000
50%       50.980000
75%       76.937500
max      100.000000
Name: amount_spent_on_dog_food, dtype: float64
Missing values: 22


## Step 10: Final Consistency Check

In [45]:
print('Dog Size:', df['dog_size'].unique())
print('Dog Gender:', df['dog_gender'].unique())
print('Dog Age Summary:', df['dog_age'].describe())

Dog Size: <StringArray>
['Small', 'Extra Large', 'Large', 'Medium', 'Extra Small', 'Unknown']
Length: 6, dtype: str
Dog Gender: <StringArray>
['Male', 'Female', 'Unknown']
Length: 3, dtype: str
Dog Age Summary: count    307.000000
mean      48.553746
std       28.343819
min        1.000000
25%       24.000000
50%       48.000000
75%       71.000000
max      100.000000
Name: dog_age, dtype: float64


## Step 11: Export Cleaned Dataset

In [46]:
output_path = '../../data/processed/consistency_dog_survey_cleaned.csv'
df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")

Saved: ../../data/processed/consistency_dog_survey_cleaned.csv
